# Physical Neumann data and nonconvex macroelements in MH and MH²M

This notebook verifies the original boundary extensions with analytical data. MH retains the Robin multiplier $\lambda=(q-p\sigma)\cdot n$ and introduces pressure moments on Neumann faces. MH²M retains separate outward conormal coefficients $A\nabla p\cdot n$ and a continuous pressure trace. Neither multiplier is silently identified with the outward physical flux in the other formulation.

The cited methods are Barrenechea, Gomes and Paredes (2024), [DOI 10.1137/22M1542556](https://doi.org/10.1137/22M1542556), and de Barros, Madureira and Valentin, [MH²M version 3](https://arxiv.org/abs/2404.16978v3). Their homogeneous-Dirichlet numerical cases are distinct from this exact boundary verification.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

import os
HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from examples.mh_campaign import l_mesh
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm import assemble
from examples.formulations.robin import define_robin, robin_gauge, recover_robin
from pymhm.fem.traces.pressure_2d import PressureTraceSpace


In [ ]:
import inspect
from examples.formulations.robin import robin_equations
print(inspect.getsource(robin_equations))
print(inspect.getsource(define_robin))


## Compatible pure Neumann patch

For $p=1+x^2+y^2$, $A=I$, $q=-2(x,y)$ and $f=-4$, prescribe $h_N=q\cdot n$ on the entire boundary. The compatibility identity is $\int_\Omega f=\int_{\partial\Omega}h_N=-4$. The volume mean is $5/3$. Two complementary L polygons cover the square; internal ear-triangulation diagonals remain local.

## Declare the three-field equations

The application declares the coupled local pressure/conormal saddle and its
global continuous pressure-trace test, using the independently chosen spaces:

$$
\begin{aligned}
A_T&=\begin{bmatrix}K_T&-\Beta_T\\-\Beta_T^{\mathsf T}&0\end{bmatrix},
&B_T&=\begin{bmatrix}0\\P_T\end{bmatrix},\\
C_T&=B_T^{\mathsf T}, &D_T&=0, &f_T&=\begin{bmatrix}F_T\\0\end{bmatrix}.
\end{aligned}
$$

Local coefficients are pressure and outward conormal eta=K grad(p).n; its
negative is physical Darcy normal flux. Gamma shares pressure nodes, while
Lambda remains private to each macrocell. The global `Equation` supplies
outward Neumann moments and fixed Gamma pressure values. Pure Neumann data add
the physical integral of the complete reconstructed pressure. Generic assembly
and solve consume these blocks directly. The shared Neumann-map kernel retains
the original boundary-mean basis for inspection and injectivity checks.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.three_field import define_three_field, recover_three_field

mesh = l_mesh(1)
exact = lambda x: 1 + np.sum(x * x, axis=1)
physical_flux = lambda x: -2 * x
neumann = {int(f): lambda x, n=mesh.normals[f]: physical_flux(x) @ n for f in mesh.boundary_faces}
common = dict(
    source=-4, dirichlet=None, neumann=neumann, mean_pressure=5 / 3, degree=2, local_refinement=4
)
with threadpool_limits(1):
    definition = define_robin(
        mesh,
        skeleton=SkeletonSpace(mesh, tuple((FaceSpace.uniform(2, 2) for _ in mesh.faces))),
        **common,
    )
    system = assemble(definition.problem)
    mh = recover_robin(
        definition, system, system.solve(constraints=robin_gauge(definition, system))
    )
    mh2m_definition = define_three_field(
        mesh,
        pressure_trace=PressureTraceSpace.uniform(mesh, 2),
        flux_space=SkeletonSpace(mesh, tuple((FaceSpace.uniform(1, 2) for _ in mesh.faces))),
        **common,
    )
    mh2m_system = assemble(mh2m_definition.problem)
    mh2m = recover_three_field(
        mh2m_definition,
        mh2m_system,
        mh2m_system.solve(constraints=pressure_constraints(mh2m_definition, mh2m_system)),
    )
checks = {}
for name, solution in (("MH", mh), ("MH2M", mh2m)):
    checks[name] = dict(
        pressure_L2=solution.l2_error(exact),
        flux_L2=solution.flux_l2_error(physical_flux),
        macro_balance=float(np.max(abs(solution.conservation_residuals()))),
    )
    assert max(checks[name].values()) < 2e-10
checks


## Five-resolution analytical campaign

The archived nonpolynomial case uses $A=\left[\begin{smallmatrix}3&0.4\\0.4&2\end{smallmatrix}\right]$ and $p=1+x+2y+\sin(\pi x)\sin(\pi y)$. All five series keep the same nonconvex geometry family and local P2 spaces. MH uses P2 Robin moments on two segments; MH²M uses continuous P2 pressure traces and P1 conormals on two segments. Mixed conditions put physical Neumann data on the right side and pressure elsewhere. Pure Neumann conditions prescribe the exact volume mean.

Error quadratures of orders eight and ten are compared independently of the assembly. Relative norms divide by the exact physical pressure or flux norm. The multilevel solve is separate from the notebook; its executed source hashes, original-equation diagnostics and dimensions are retained.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((ROOT / "examples/results/mh/boundary-comparison.json").read_text())
    assert not record["source_changed_during_run"]
    assert len(record["rows"]) == 25
    [{k: r[k] for k in ("method", "boundary", "resolution", "pressure_relative", "flux_relative")}
     for r in record["rows"] if r["resolution"] == max(x["resolution"] for x in record["rows"])]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(ROOT / "docs/figures/mh/boundary-convergence.png")))


The pressure/flux convergence and independent UFL block comparisons verify these boundary and geometry implementations. They do not establish the article's global Dirichlet SPD result for the augmented Neumann MH system, nor uniform stability for arbitrary independently chosen MH²M face spaces.